# HW2. Ускорение обучения Qwen3 на одной GPU

## 0. Окружение


In [1]:
import sys

!{sys.executable} -m pip install torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu124
!{sys.executable} -m pip install transformers==4.56.2 datasets==3.6.0 accelerate==1.6.0 trl==0.23.1 liger-kernel==0.6.2 pandas packaging ninja psutil wheel
# flash-attn сам найдёт готовый wheel под torch 2.6 + cu12 на GitHub и не будет компилироваться
!{sys.executable} -m pip install flash-attn==2.7.3 --no-build-isolation

...
...
...


In [2]:
import torch, transformers, trl, flash_attn

print(torch.__version__, transformers.__version__, trl.__version__, flash_attn.__version__)
print(torch.cuda.get_device_name())

2.6.0+cu124 4.56.2 0.23.1 2.7.3
NVIDIA H100 PCIe


## 1. Конфигурации



In [3]:
from experiments import EXPERIMENTS

for name, config in EXPERIMENTS.items():
    print(f"{name:30} {config}")

baseline                       {'seconds': 300}
bs16_acc4                      {'seconds': 90, 'per_device_train_batch_size': 16, 'gradient_accumulation_steps': 4}
checkpointing_bs64             {'seconds': 90, 'gradient_checkpointing': True, 'per_device_train_batch_size': 64, 'gradient_accumulation_steps': 1}
offloading                     {'seconds': 90, 'activation_offloading': True}
liger                          {'seconds': 90, 'use_liger_kernel': True}
liger_bs64                     {'seconds': 90, 'use_liger_kernel': True, 'per_device_train_batch_size': 64, 'gradient_accumulation_steps': 1}
compile                        {'seconds': 90, 'torch_compile': True}
fa2                            {'seconds': 90, 'attention': 'flash_attention_2'}
fa2_padding_free               {'seconds': 90, 'attention': 'flash_attention_2', 'padding_free': True}
fa2_packing                    {'seconds': 90, 'attention': 'flash_attention_2', 'packing': True}
fa2_packing_bs6                {'seconds': 

## 2. Бейзлайн



In [10]:
import sys

!PYTHONPATH=. {sys.executable} run.py baseline

Подготовлено 5000/50500 статей
...
Подготовлено 50000/50500 статей
Saving the dataset (1/1 shards): 100%|█| 244768/244768
Saving the dataset (1/1 shards): 100%|█| 2928/2928
{"step": 10, "elapsed_seconds": 10.156903528608382, "step_seconds": 0.5937535092234612, "target_tokens": 5417, "warmup": true}
{'loss': 11.2275, 'grad_norm': 5.90625, 'learning_rate': 1.3499999999999998e-05, 'epoch': 0.002614720878546215}
...
{"step": 500, "elapsed_seconds": 295.1452105809003, "step_seconds": 0.5974772395566106, "target_tokens": 5592, "warmup": false}
{'loss': 6.3482, 'grad_norm': 0.8984375, 'learning_rate': 0.0003, 'epoch': 0.13073604392731075}
{'train_runtime': 300.6262, 'train_loss': 7.617337719630633, 'epoch': 0.13308929271800235}
{
  "train_seconds": 300.6254651704803,
  "optimizer_steps": 509,
  "target_tokens": 2914037,
  "tokens_per_second_including_warmup": 9693.247371268073,
  "warmup_steps": 10,
  "measured_steps": 499,
  "measured_seconds": 290.46856164187193,
  "tokens_per_second": 9834

## 3. Короткие эксперименты (90 с)

In [11]:
short = [name for name in EXPERIMENTS if name != "baseline"]
!mkdir -p ./logs

for name in short:
    print("=" * 20, name)
    !PYTHONPATH=. {sys.executable} run.py {name} > ./logs/{name}.log 2>&1; tail -n 4 ./logs/{name}.log

==================== bs16_acc4
  "peak_reserved_gib": 26.04296875,
  "validation_loss": 7.752227146743002,
  "validation_target_tokens": 50694
}
==================== checkpointing_bs64
  "peak_reserved_gib": 27.337890625,
  "validation_loss": 8.100458230587389,
  "validation_target_tokens": 50694
}
==================== offloading
    _engine_run_backward(
  File "/home/jovyan/work/.lib/lib/python3.10/site-packages/torch/autograd/graph.py", line 823, in _engine_run_backward
    return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
RuntimeError: attn_bias is not correctly aligned (strideH) .attn_bias.stride(1) = 35721, and should be a multiple of 4.
==================== liger
  "peak_reserved_gib": 28.501953125,
  "validation_loss": 7.887530114799627,
  "validation_target_tokens": 50694
}
==================== liger_bs64
  "peak_reserved_gib": 45.208984375,
  "validation_loss": 7.886738270515285,
  "validation_target_tokens": 50694
}
=======

## 4. Таблица результатов

Собираем все `./results/*/summary.json`. S считается к бейзлайну Таблица сохраняется в ./results_table.csv

In [12]:
import json
from pathlib import Path

import pandas as pd

rows = []
for path in sorted(Path("./results").glob("*/summary.json")):
    rows.append(dict(config=path.parent.name, **json.loads(path.read_text())))

df = pd.DataFrame(rows).set_index("config")
df["S"] = df.tokens_per_second / df.loc["baseline", "tokens_per_second"]

columns = ["train_seconds", "tokens_per_second", "S", "peak_allocated_gib", "peak_reserved_gib",
           "validation_loss", "optimizer_steps", "target_tokens"]
df = df[columns].sort_values("tokens_per_second", ascending=False)
df.to_csv("./results_table.csv")
df.round(3)

,train_seconds,tokens_per_second,S,peak_allocated_gib,peak_reserved_gib,validation_loss,optimizer_steps,target_tokens
config,,,,,,,,
fa2_packing,90.606,31617.222,3.215,40.196,76.232,8.052,89,2848733
fa2_padding_free_liger_bs64,90.092,29367.958,2.986,14.193,24.773,6.248,453,2589220
fa2_packing_bs6,90.004,27259.167,2.772,13.443,14.232,6.360,406,2436727
fa2_padding_free,90.204,26775.262,2.723,15.155,19.727,6.380,420,2394900
fa2_padding_free_liger,90.083,23820.917,2.422,12.335,14.031,6.615,364,2071941
bs16_acc4,90.031,11244.043,1.143,24.259,26.043,7.752,176,999165
fa2,90.029,10949.136,1.113,39.342,41.037,7.788,170,965083
liger,90.327,10447.328,1.062,26.880,28.502,7.888,153,868309
liger_bs64,90.202,9852.178,1.002,43.954,45.209,7.887,152,863000


## 5. Итоговый запуск (300 с)

In [ ]:
import re

best = df.drop(index="baseline").tokens_per_second.idxmax()
print("лучший:", best, EXPERIMENTS[best])

changes = dict(EXPERIMENTS[best])
changes.pop("seconds")
attention = changes.pop("attention", "sdpa")
changes["output_dir"] = "./results/final"

text = Path("./train.py").read_text()
text = text.replace('ATTENTION_IMPLEMENTATION: str = "sdpa"', f'ATTENTION_IMPLEMENTATION: str = "{attention}"')
for key, value in changes.items():
    text = re.sub(rf"^    {key}=.*,$", f"    {key}={value!r},", text, flags=re.MULTILINE)

Path("./final").mkdir(exist_ok=True)
Path("./final/train.py").write_text(text)
!diff ./train.py ./final/train.py

лучший: fa2_packing {'seconds': 90, 'attention': 'flash_attention_2', 'packing': True}
29c29
< ATTENTION_IMPLEMENTATION: str = "sdpa"
---
> ATTENTION_IMPLEMENTATION: str = "flash_attention_2"
42c42
<     output_dir="results/baseline",
---
>     output_dir='./results/final',
50c50
<     packing=False,
---
>     packing=True,


In [14]:
!{sys.executable} ./final/train.py > ./logs/final.log 2>&1; tail -n 16 ./logs/final.log
!cp ./results/final/summary.json ./final/summary.json

final = json.loads(Path("./final/summary.json").read_text())
baseline = json.loads(Path("./results/baseline/summary.json").read_text())
S = final["tokens_per_second"] / baseline["tokens_per_second"]
print(f"baseline {baseline['tokens_per_second']:.0f} tok/s, final {final['tokens_per_second']:.0f} tok/s, S = {S:.2f}")

{'train_runtime': 300.979, 'train_loss': 7.73822174582593, 'epoch': 0.4339622641509434}
{
  "train_seconds": 300.9781901547685,
  "optimizer_steps": 299,
  "target_tokens": 9560433,
  "tokens_per_second_including_warmup": 31764.537473907494,
  "warmup_steps": 10,
  "measured_steps": 289,
  "measured_seconds": 290.5284996125847,
  "tokens_per_second": 31807.024138156936,
  "seconds_per_optimizer_step": 1.0052889259951028,
  "peak_allocated_gib": 40.22846221923828,
  "peak_reserved_gib": 76.232421875,
  "validation_loss": 5.682788388387745,
  "validation_target_tokens": 50694
}
baseline 9834 tok/s, final 31807 tok/s, S = 3.23


выводы в Readme